# Goldfish Detection - YOLOv8 Training
## Fish Guardian Project - Custom Model Training

This notebook trains a custom YOLOv8 model to detect goldfish in your aquarium.

**Hardware:** Google Colab Free GPU (T4)

**Training Time:** ~15-30 minutes for 100 epochs

## Step 1: Check GPU Availability

In [ ]:
# Check if GPU is available
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ WARNING: No GPU detected! Go to Runtime > Change runtime type > GPU")

## Step 2: Install YOLOv8 (Ultralytics)

In [ ]:
# Install Ultralytics YOLOv8
!pip install ultralytics -q

# Verify installation
from ultralytics import YOLO
import ultralytics
print(f"✅ Ultralytics version: {ultralytics.__version__}")

## Step 3: Download Your Dataset from Roboflow

In [ ]:
# Download your goldfish dataset
!curl -L "https://app.roboflow.com/ds/UzlM23koMD?key=CcVpLVu22Y" > roboflow.zip
!unzip -q roboflow.zip
!rm roboflow.zip

print("✅ Dataset downloaded and extracted!")

# Show dataset structure
!ls -la

## Step 4: Verify Dataset Configuration

In [ ]:
# Check the data.yaml file
!cat data.yaml

# Count images in each split
import os
print("\n📊 Dataset Summary:")
print(f"Train images: {len(os.listdir('train/images'))}")
print(f"Valid images: {len(os.listdir('valid/images'))}")
print(f"Test images: {len(os.listdir('test/images'))}")
print(f"\nTrain labels: {len(os.listdir('train/labels'))}")
print(f"Valid labels: {len(os.listdir('valid/labels'))}")
print(f"Test labels: {len(os.listdir('test/labels'))}")

## Step 5: Train YOLOv8n Model

**Training Parameters:**
- **Model:** YOLOv8n (nano - fastest, optimized for Edge TPU)
- **Epochs:** 100 (will take ~20-30 minutes)
- **Image size:** 640x640
- **Batch size:** 16 (adjust if GPU memory issues)
- **Device:** GPU (CUDA)

In [ ]:
from ultralytics import YOLO
import time

# Load pretrained YOLOv8n model
model = YOLO('yolov8n.pt')

print("🚀 Starting training...")
print("This will take approximately 20-30 minutes.\n")

start_time = time.time()

# Train the model
results = model.train(
    data='data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,  # GPU
    project='goldfish_training',
    name='yolov8n_goldfish',
    patience=20,  # Early stopping if no improvement for 20 epochs
    save=True,
    save_period=10,  # Save checkpoint every 10 epochs
    cache=True,  # Cache images for faster training
    plots=True,  # Generate training plots
)

training_time = (time.time() - start_time) / 60
print(f"\n✅ Training complete in {training_time:.1f} minutes!")

## Step 6: View Training Results

In [ ]:
# Display training metrics
from IPython.display import Image, display

print("📊 Training Results:\n")

# Show confusion matrix
print("Confusion Matrix:")
display(Image('goldfish_training/yolov8n_goldfish/confusion_matrix.png', width=600))

# Show training curves
print("\nTraining Curves:")
display(Image('goldfish_training/yolov8n_goldfish/results.png', width=800))

# Show sample predictions
print("\nValidation Batch Predictions:")
display(Image('goldfish_training/yolov8n_goldfish/val_batch0_pred.png', width=800))

## Step 7: Validate Model Performance

In [ ]:
# Load best trained model
best_model = YOLO('goldfish_training/yolov8n_goldfish/weights/best.pt')

# Validate on test set
print("🔍 Validating model on test set...\n")
metrics = best_model.val(data='data.yaml', split='test')

# Print key metrics
print(f"\n📈 Test Set Performance:")
print(f"mAP50: {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.p[0]:.3f}")
print(f"Recall: {metrics.box.r[0]:.3f}")

# Interpretation
print("\n💡 What these metrics mean:")
print(f"- mAP50 > 0.8 = Excellent detection")
print(f"- mAP50 > 0.6 = Good detection")
print(f"- mAP50 < 0.5 = Needs more training data or epochs")

## Step 8: Test Inference Speed

In [ ]:
# Test inference speed on GPU
import time
import cv2
import numpy as np

# Create dummy image
test_img = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)

# Warm-up
for _ in range(10):
    best_model(test_img, verbose=False)

# Benchmark
num_runs = 100
start = time.time()
for _ in range(num_runs):
    best_model(test_img, verbose=False)
elapsed = time.time() - start

avg_time = (elapsed / num_runs) * 1000  # ms
fps = num_runs / elapsed

print(f"⚡ GPU Inference Speed:")
print(f"Average: {avg_time:.1f}ms per image")
print(f"FPS: {fps:.1f}")
print(f"\n💡 On Raspberry Pi with Coral TPU, expect: 20-30 FPS")

## Step 9: Export Model for EdgeTPU

This exports the model to TFLite format, which we'll convert to EdgeTPU format on the Raspberry Pi.

In [ ]:
# Export to TFLite (INT8 quantization for EdgeTPU)
print("📦 Exporting model to TFLite format for EdgeTPU...\n")

# Export with INT8 quantization
best_model.export(
    format='tflite',
    imgsz=640,
    int8=True,  # Required for EdgeTPU
    data='data.yaml'  # Use dataset for quantization calibration
)

print("✅ Model exported!")
print("\nExported files:")
!ls -lh goldfish_training/yolov8n_goldfish/weights/

## Step 10: Download Trained Model

Download these files to your computer:
1. `best.pt` - PyTorch model (for testing)
2. `best_saved_model/` - TFLite model directory (for EdgeTPU conversion)

In [ ]:
# Create zip file for download
print("📦 Creating download package...\n")

!mkdir -p goldfish_model_export
!cp goldfish_training/yolov8n_goldfish/weights/best.pt goldfish_model_export/
!cp -r goldfish_training/yolov8n_goldfish/weights/best_saved_model goldfish_model_export/
!cp goldfish_training/yolov8n_goldfish/results.png goldfish_model_export/
!cp goldfish_training/yolov8n_goldfish/confusion_matrix.png goldfish_model_export/

!zip -r goldfish_model.zip goldfish_model_export/

print("✅ Package ready!")
print("\n📥 Download 'goldfish_model.zip' from the Files panel (left sidebar)")
print("\nThe zip contains:")
print("  - best.pt (PyTorch model)")
print("  - best_saved_model/ (TFLite model for EdgeTPU)")
print("  - results.png (training curves)")
print("  - confusion_matrix.png (performance metrics)")

!ls -lh goldfish_model.zip

## 🎉 Training Complete!

### Next Steps:

1. **Download `goldfish_model.zip`** from the Files panel
2. **Transfer to Raspberry Pi** (we'll help with this)
3. **Convert to EdgeTPU format** using the Coral compiler
4. **Integrate into fish-guardian** system
5. **Test live detection** on your aquarium!

### Expected Performance:
- **Raspberry Pi CPU:** 1-2 FPS (too slow)
- **Coral USB TPU:** 20-30 FPS (perfect!)
- **Accuracy:** Should detect goldfish with 85-95% confidence

---

**Project:** Fish Guardian - AI Upgrade (Phase 2)

**Training Dataset:** 300 annotated goldfish images

**Model:** YOLOv8n (optimized for edge devices)